# Task 4 — Email Spam Detection with Machine Learning

**Oasis Infobyte Data Science Internship**

This notebook demonstrates a complete beginner-friendly NLP classification workflow.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix


In [ ]:
df = pd.read_csv("../data/spam.csv")
df.head()

In [ ]:
df.info()
print("\nClass counts:")
print(df["label"].value_counts())

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(data=df, x="label")
plt.title("Message Class Distribution")
plt.show()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(df["message"], df["label"], test_size=0.20, random_state=42, stratify=df["label"])
print(len(X_train), len(X_test))

In [ ]:
models = {
 "Multinomial Naive Bayes": Pipeline([("tfidf", TfidfVectorizer(stop_words="english", ngram_range=(1,2))), ("model", MultinomialNB())]),
 "Logistic Regression": Pipeline([("tfidf", TfidfVectorizer(stop_words="english", ngram_range=(1,2))), ("model", LogisticRegression(max_iter=1000, random_state=42))])
}
rows=[]
for name, model in models.items():
    model.fit(X_train,y_train)
    pred=model.predict(X_test)
    rows.append([name,accuracy_score(y_test,pred),precision_score(y_test,pred,pos_label="spam"),recall_score(y_test,pred,pos_label="spam"),f1_score(y_test,pred,pos_label="spam")])
results=pd.DataFrame(rows,columns=["Model","Accuracy","Precision","Recall","F1"]); results

In [ ]:
best_name=results.sort_values("F1",ascending=False).iloc[0]["Model"]
best=models[best_name]
pred=best.predict(X_test)
print("Best model:",best_name)
print(classification_report(y_test,pred))

In [ ]:
cm=confusion_matrix(y_test,pred,labels=["ham","spam"])
sns.heatmap(cm,annot=True,fmt="d",xticklabels=["Ham","Spam"],yticklabels=["Ham","Spam"])
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Confusion Matrix"); plt.show()

In [ ]:
samples=["Congratulations! You won a free cash prize. Claim now!", "Can you send me the notes from class today?"]
for msg in samples:
    print(msg, "=>", best.predict([msg])[0])